# KEN Model — Data Cleaning

Prepares input data for the KEN (KAUST Economy-Nature) model: it converts the desalination
cost time series to model units, maps water-use intensities from 21 to 85 sectors, and
processes the raw exports CSV into the model's input format.

In [ ]:
%reload_ext autoreload
%autoreload 2
from model.parameters_baseline import ModelParameters
from model.calibration import ParametersCalibrated
import pandas as pd
import csv
from model.utils import to_float_or_nan

In [3]:
parameters = ModelParameters.default_values()
pc = ParametersCalibrated(parameters)
sectors_io = list(pc.sectors)

122.275
2010000.0
tSAR capex/m3 cap 6000000.0
0.0014266909502320302
4289.151135


In [6]:
pd.read_csv('input_data/water_use/ts_desal_2.csv', index_col=0, delimiter=";").map(to_float_or_nan)

,Capacity (Mm3/d),Cumulative Capacity (Mm3/d),CapEx (M SAR),OpEx (M SAR/year)
Online date,,,,
1979,0.001763,0.001763,10.578,0.965243
1982,0.003000,0.004763,18.000,2.607743
1984,0.002000,0.006763,12.000,3.702742
1985,0.000600,0.007363,3.600,4.031243
1993,0.004500,0.011863,27.000,6.494993
1994,0.012834,0.024697,77.004,13.521608
1996,0.150351,0.175048,902.106,95.838780
1997,0.422780,0.597828,2536.680,327.310830
1998,0.222287,0.820115,1333.722,449.012963


In [20]:
df = pd.read_csv('input_data/water_use/ts_desal.csv', index_col=0, header=0, delimiter=';').map(to_float_or_nan)
df["cap_ex"] = df["cap_ex"] / 1000000
df["op_ex"] = df["op_ex"] / 1000000
df.to_csv("input_data/water_use/ts_desal_tSAR.csv", index=True)

## Adapt water data from 21 to 85 sectors

In [3]:
# Load water use data for 21 sectors
df = pd.read_csv('input_data/water_use/WU_intensity_21_TO.csv', index_col=1, header=0)
df.drop(df.columns[0], axis=1, inplace=True)
df = df.transpose()

# Rename columns to match the sectors in the model
rename_dict = {
    'Sewage': 'Sewerage',
    'Other mining and quarrying': 'Other mining and quarrying activities',
    'Mining support services': 'Mining support service activities',
    'Manufacture of wood and of products of wood and cork, except furniture;\nmanufacture of articles of straw and plaiting materials': 'Manufacture of woods, wood products and cork except furniture',
    'Manufacture of pharmaceuticals, medicinal chemical and botanical products': 'Manufacture of basic pharmaceutical products and pharmaceutical preparations',
    'Manufacture of machinery and equipment n.e.c': 'Manufacture of machinery and equipment n.e.c.',
    'Other manufacturing industries': 'Other manufacturing',
    'Supplies of electricity, gas, steam and air conditioning': 'Electricity, gas, steam and air conditioning supply',
    'Waste collection, treatment & disposal activities; materials recovery': 'Waste collection, treatment and disposal activities; materials recovery'
}
df.rename(columns=rename_dict, inplace=True)

# Create new dataframe with pc.sectors as columns and feed in data
wu = pd.DataFrame(columns=pc.sectors)
wu.loc[0] = [0.] * len(pc.sectors)
wu.index = pd.Index(['water_use'])
for sector in pc.sectors:
    if sector in df.columns:
        wu.at['water_use', sector] = df.at['WU_intensity_21', sector]

# Fill average values for 3 agriculture sectors
agri_sectors = pc.sectors[0:3]
for sector in agri_sectors:
    wu.at['water_use', sector] = df.at['WU_intensity_21', "Agriculture"] #/ 3

# Fill average values for 49 service sectors
agri_sectors = pc.sectors[36:85] 
for sector in agri_sectors:
    wu.at['water_use', sector] = df.at['WU_intensity_21', "Services"] #/ 49

wu.to_csv("input_data/water_use/WU_intensity_85_TO.csv", index=True)

In [ ]:
## Process the CSV raw data

# Read the raw CSV file
input_file = 'input_data/ts_exports_raw.csv'
df = pd.read_csv(input_file)

# Remove any leading/trailing whitespace from column names
df.columns = df.columns.str.strip()

# Remove any leading/trailing whitespace from SITC Section values
df['SITC Section'] = df['SITC Section'].str.strip()

# Remove quotes from column names and values
df.columns = df.columns.str.replace('"', '')
df = df.replace('"', '', regex=True)

# Convert the data to numeric, handling commas and coercing errors
for col in df.columns[1:]:
    df[col] = pd.to_numeric(df[col].str.replace(',', ''), errors='coerce')

# Pivot the dataframe to have years as rows and SITC Sections as columns
df_pivoted = df.set_index('SITC Section').transpose()


# Write the processed data to a new CSV file
output_file = 'input_data/ts_exports_automatic.csv'
df_pivoted.to_csv(output_file, index=False)

print(f"Processed data has been saved to {output_file}")